In [ ]:
!pip install vllm

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import subprocess
import polars as pl
import torch
from google.colab import userdata  # Pro Google Colab
from kaggle_secrets import UserSecretsClient  # Pro Kaggle
from huggingface_hub import login, HfApi, DatasetCard, DatasetCardData
from datasets import load_dataset
from datasets import Dataset, DatasetDict
from vllm import LLM, SamplingParams

from tqdm import tqdm





### **1. LOAD SECRETS (Kaggle/Colab Fallback) & HF LOGIN**

In [ ]:
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    user_email = user_secrets.get_secret("user_email")
    user_name = user_secrets.get_secret("user_name")
except Exception:
    try:
        hf_token = userdata.get("HF_TOKEN")
        user_email = userdata.get("USER_EMAIL")
        user_name = userdata.get("USER_NAME")
    except Exception as e:
        print(f"Warning: Failed to load secrets automatically: {e}")

In [ ]:
if hf_token:
    login(token=hf_token)
    print("Successfully logged into Hugging Face Hub!")
else:
    raise ValueError("HF_TOKEN was not found in Secrets!")

### **2. GIT CONFIGURATION***

In [ ]:
def set_git_config(email, name):
    try:
        subprocess.run(
            ["git", "config", "--global", "user.email", email], check=True
        )
        subprocess.run(["git", "config", "--global", "user.name", name], check=True)
        print(f"Git configuration set for: {name} <{email}>")
    except subprocess.CalledProcessError as e:
        print(f"Error setting Git configuration: {e}")

In [ ]:
if user_email and user_name:
    set_git_config(user_email, user_name)

### **3. SETUP REPO_ID AND INITIALIZATION**

In [ ]:
api = HfApi()
repo_exists = False
hf_username = "KRadim"

In [ ]:
DATASET_REPO_ID = f"{hf_username}/multilingual-text2cypher-2025"

### **4. LOADING AND MERGING THE ORIGINAL DATASET (TRAIN + TEST) INTO POLARS**

In [ ]:
ds = load_dataset("neo4j/text2cypher-2025v1")
df_raw_train = pl.from_pandas(ds["train"].to_pandas())

In [ ]:
if "test" in ds:
    df_raw_test = pl.from_pandas(ds["test"].to_pandas())
    df_raw = pl.concat([df_raw_train, df_raw_test])
    print(f"Merged! Train: {len(df_raw_train)}, Test: {len(df_raw_test)} | Total to process: {len(df_raw)}")
else:
    df_raw = df_raw_train
    print(f"There is no test split in the dataset; I am proceeding only with the train split ({len(df_raw)} rows).")

### **5. INITIALIZE vLLM MODEL**

In [ ]:
max_seq_length = 2048  # Maximum context length
model_name = "google/gemma-4-E4B-it"

In [ ]:
print(f"\n--> Loading vLLM engine for {model_name}...")

#### *Initializing the vLLM engine*

In [ ]:
llm = LLM(
    model=model_name,
    max_model_len=max_seq_length,
    tensor_parallel_size=2,  # Change to 2 if you want to split the model across 2 GPUs
    gpu_memory_utilization=0.90,
    trust_remote_code=True
)

#### *Setting parameters for deterministic translation*

In [ ]:
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=128
)

In [ ]:
tokenizer = llm.get_tokenizer()

In [ ]:
def translate_vllm_batch(questions: list[str], target_language: str) -> list[str]:
    """Generates translations for the entire specified list using continuous batching in vLLM."""
    
    messages_batch = [
        [
            {
                "role": "system",
                "content": (
                    f"You are a professional technical translator. Translate the following user question "
                    f"into {target_language}. Maintain exact technical terms, entity names, and schema context. "
                    f"Provide ONLY the direct translation, without any introduction or markdown formatting."
                )
            },
            {"role": "user", "content": q}
        ]
        for q in questions
    ]

    # Prompt formatting via chat template
    prompts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        for msg in messages_batch
    ]

    # vLLM automatically handles massive batches at once thanks to Continuous Batching & PagedAttention
    outputs = llm.generate(prompts, sampling_params, show_progress_bar=True)
    
    return [out.outputs[0].text.strip() for out in outputs]

### **6. ITERATIVE CHUNK PROCESSING (CHUNK_SIZE = 500)**

In [ ]:
CHECKPOINT_PATH = "multilingual_checkpoint.parquet"
CHUNK_SIZE = 5000

In [ ]:
TARGET_LANGUAGES = {
    "question_cz": "Czech",
    "question_it": "Italian",
    "question_de": "German",
    "question_es": "Spanish",
    "question_fr": "French"
}

In [ ]:
if os.path.exists(CHECKPOINT_PATH):
    print(f"🔄 Loading existing checkpoint from '{CHECKPOINT_PATH}'...")
    df_multilingual = pl.read_parquet(CHECKPOINT_PATH)
else:
    print("🆕 I am creating a new structure for the translated data...")
    df_multilingual = df_raw.rename({"question": "question_en"})
    for col_name in TARGET_LANGUAGES.keys():
        df_multilingual = df_multilingual.with_columns(
            pl.lit(None, dtype=pl.Utf8).alias(col_name)
        )

In [ ]:
total_rows = len(df_multilingual)

In [ ]:
print("\n--> Generating translations in chunks...")

In [ ]:
for chunk_start in range(0, total_rows, CHUNK_SIZE):
    chunk_end = min(chunk_start + CHUNK_SIZE, total_rows)
    print(f"\n==================================================")
    print(f"🚀 Chunk: rows {chunk_start} to {chunk_end} of {total_rows}")
    print(f"==================================================")

    chunk_updated = False

    for col_name, lang_name in TARGET_LANGUAGES.items():
        chunk_questions = df_multilingual["question_en"][chunk_start:chunk_end].to_list()
        current_values = df_multilingual[col_name][chunk_start:chunk_end].to_list()

        # Check if a translation is missing in the given chunk
        needs_translation = any(v is None for v in current_values)
        if not needs_translation:
            print(f"⏩ Column '{col_name}' for this chunk is already done, skipping...")
            continue

        print(f"Generating translation for {lang_name} ({col_name})...")
        translated_chunk = translate_vllm_batch(chunk_questions, target_language=lang_name)

        # Update Polars DataFrame
        updated_col = df_multilingual[col_name].to_list()
        updated_col[chunk_start:chunk_end] = translated_chunk
        df_multilingual = df_multilingual.with_columns(
            pl.Series(name=col_name, values=updated_col)
        )
        chunk_updated = True

    # Checkpoint after each solved chunk
    if chunk_updated:
        df_multilingual.write_parquet(CHECKPOINT_PATH)
        print(f"✅ Chunk {chunk_start} to {chunk_end} saved to '{CHECKPOINT_PATH}'!")

### **7. UNPIVOT (MELT), SHUFFLE, AND NEW TRAIN/TEST SPLIT**

In [ ]:
print("\n--> Transforming data in Polars (Unpivot/Melt + Shuffle)...")

In [ ]:
df_flat = df_multilingual.unpivot(
    on=["question_en", "question_cz", "question_it", "question_de", "question_es", "question_fr"],
    index=["cypher", "schema", "data_source", "instance_id"],
    variable_name="language",
    value_name="question"
).with_columns(
    pl.col("language").str.replace("question_", "")
)

In [ ]:
df_final = df_flat.sample(fraction=1.0, shuffle=True, seed=42)

In [ ]:
test_size = int(len(df_final) * 0.10)
df_test = df_final.head(test_size)
df_train = df_final.tail(len(df_final) - test_size)

In [ ]:
print(f"Total rows: {len(df_final)} | Train: {len(df_train)} | Test: {len(df_test)}")

### **8. UPLOAD TO HF HUB**

In [ ]:
print(f"\n--> Uploading dataset to repository: {DATASET_REPO_ID}...")

In [ ]:
dataset_dict = DatasetDict({
    "train": Dataset.from_polars(df_train),
    "test": Dataset.from_polars(df_test)
})

In [ ]:
dataset_dict.push_to_hub(
    repo_id=DATASET_REPO_ID,
    token=hf_token,
    private=False
)

In [ ]:
print("\n--> Creating and uploading Dataset Card (README.md)...")

#### *Upload Dataset Card (README)*

In [ ]:
card_metadata = DatasetCardData(
    language=["en", "cs", "it", "de", "es", "fr"],
    license="apache-2.0",
    tags=["text-to-cypher", "neo4j", "multilingual", "synthetic", "gemma-4", "graph-db"],
    dataset_info={
        "features": [
            {"name": "language", "dtype": "string"},
            {"name": "question", "dtype": "string"},
            {"name": "cypher", "dtype": "string"},
            {"name": "schema", "dtype": "string"},
            {"name": "data_source", "dtype": "string"},
            {"name": "instance_id", "dtype": "string"},
        ],
        "splits": [
            {"name": "train", "num_bytes": dataset_dict["train"].size_in_bytes, "num_examples": len(dataset_dict["train"])},
            {"name": "test", "num_bytes": dataset_dict["test"].size_in_bytes, "num_examples": len(dataset_dict["test"])},
        ],
    }
)

In [ ]:
dataset_card_text = f"""
# Multilingual Text-to-Cypher Dataset (2025)

This dataset is an expanded, **multilingual synthetic edition** created by combining all original splits of [`neo4j/text2cypher-2025v1`](https://huggingface.co/datasets/neo4j/text2cypher-2025v1).

It is designed for fine-tuning Large Language Models (LLMs) to generate accurate **Neo4j Cypher queries** from natural language prompts across 6 languages:
- 🇬🇧 **English (`en`)**
- 🇨🇿 **Czech (`cz`)**
- 🇮🇹 **Italian (`it`)**
- 🇩🇪 **German (`de`)**
- 🇪🇸 **Spanish (`es`)**
- 🇫🇷 **French (`fr`)**

---

## Dataset Splits

- **`train`**: {len(dataset_dict['train'])} examples (90%)
- **`test`**: {len(dataset_dict['test'])} examples (10%)

---

## Dataset Structure

All original data splits were combined, translated into 5 additional target languages, unpivoted, and globally reshuffled into balanced train/test splits.

### Fields:
- `language`: Language code (`en`, `cz`, `it`, `de`, `es`, `fr`).
- `question`: Natural language question in the specified language.
- `cypher`: Corresponding Neo4j Cypher query.
- `schema`: Database graph schema definition.
- `data_source`: Source database reference.
- `instance_id`: Unique identifier for the base prompt.

---

## Creation Process

1. **Data Concatenation**: Combined all split data from `neo4j/text2cypher-2025v1`.
2. **Translation & Synthesis**: Translated queries into CZ, IT, DE, ES, and FR using **`gemma-4-E4B-it`**.
3. **Data Transformation**: Unpivoted using **Polars** (expanding row count 6x).
4. **Reshuffling & Splitting**: Globally shuffled with seed `42` and split into new `train` (90%) and `test` (10%) splits.

---

## Usage in Python / Hugging Face

```python
from datasets import load_dataset

dataset = load_dataset("{DATASET_REPO_ID}")

train_data = dataset["train"]
test_data = dataset["test"]

print("Train sample:", train_data[0])
print("Test sample:", test_data[0])
"""

In [ ]:
card = DatasetCard(dataset_card_text)
card.data = card_metadata
card.push_to_hub(repo_id=DATASET_REPO_ID, token=hf_token)

In [ ]:
print("\n" + "=" * 60)
print(f"✅ Everything successfully completed!")
print(f"The dataset is available at: https://huggingface.co/datasets/{DATASET_REPO_ID}")
print("=" * 60)